# Hierarchical Forecasting

Wiki reference for [hierarchical forecasting](https://ml-viz-ruby.vercel.app/wiki/hierarchical-forecasting).

**The idea in one sentence.** When you forecast a hierarchy (national → regions → stores)
independently, the numbers **don't add up** — the store forecasts won't sum to the national one;
**reconciliation** (MinT) projects the base forecasts onto the coherent subspace, restoring
consistency *and* usually improving accuracy by borrowing strength across levels.

We build the summing matrix and MinT reconciliation from scratch, **validate that
reconciliation restores coherence and reduces error**, then cover the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
rng = np.random.default_rng(5)

## 1 — Summing matrix and coherence check

In [ ]:
# Hierarchy: 1 national → 2 regions → 4 stores
# S encodes: national = sum(all), region_A = s1+s2, region_B = s3+s4, plus 4 bottom series
S = np.array([
    [1, 1, 1, 1],   # national
    [1, 1, 0, 0],   # region A
    [0, 0, 1, 1],   # region B
    [1, 0, 0, 0],   # store 1
    [0, 1, 0, 0],   # store 2
    [0, 0, 1, 0],   # store 3
    [0, 0, 0, 1],   # store 4
], dtype=float)

# True bottom-level values
b_true = np.array([10., 15., 8., 20.])
y_true = S @ b_true
print("True coherent forecasts:", y_true)

# Base forecasts (incoherent)
b_hat = b_true + rng.normal(0, 2, 4)  # add noise
y_hat = np.zeros(7)
y_hat[:3] = y_true[:3] + rng.normal(0, 3, 3)  # aggregate forecasts also noisy
y_hat[3:] = b_hat
print("Base (incoherent) forecasts:", y_hat.round(2))
print("Coherence error (national vs sum of stores):", abs(y_hat[0] - y_hat[3:].sum()).round(3))

## 2 — OLS MinT reconciliation

In [ ]:
def mint_ols(y_hat, S):
    """MinT reconciliation with W=I (OLS version)."""
    # ỹ = S (S'S)^{-1} S' y_hat = P y_hat
    P = S @ np.linalg.inv(S.T @ S) @ S.T
    return P @ y_hat

y_tilde = mint_ols(y_hat, S)
print("Reconciled forecasts (MinT-OLS):", y_tilde.round(2))
print("Coherence error after reconciliation:", abs(y_tilde[0] - y_tilde[3:].sum()).round(6))
print("Forecast error vs true:")
print("  Before:", np.abs(y_hat - y_true).mean().round(3))
print("  After: ", np.abs(y_tilde - y_true).mean().round(3))

### Validate: reconciliation restores coherence and cuts error

The base forecasts are **incoherent** (national $\ne$ sum of stores). MinT reconciliation makes
them coherent by construction, and — because it pools information across levels — the reconciled
forecasts are also **more accurate** than the base ones. We confirm both.

In [ ]:
coh_after = abs(y_tilde[0] - y_tilde[3:].sum())
err_before = np.abs(y_hat - y_true).mean()
err_after = np.abs(y_tilde - y_true).mean()
print(f'coherence error after reconciliation: {coh_after:.6f}')
print(f'mean forecast error: before {err_before:.3f} -> after {err_after:.3f}')
assert coh_after < 1e-6, 'reconciled forecasts are coherent (national = sum of stores)'
assert err_after <= err_before, 'reconciliation does not worsen accuracy — here it improves it'
print('\n✅ MinT makes the hierarchy add up AND borrows strength across levels to cut error')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **incoherent forecasts** | levels contradict each other (demo) — reconcile |
| **bottom-up only** | coherent but throws away aggregate-level signal |
| **top-down only** | needs stable disaggregation proportions |
| **MinT weight matrix** | OLS ($W=I$) is simplest; a better $W$ needs error covariances |
| **large hierarchies** | reconciliation cost grows with the number of series |

Demo: base forecasts don't add up; reconciliation makes them coherent.

In [ ]:
# The problem reconciliation solves, made explicit: forecasting each level INDEPENDENTLY gives
# numbers that contradict each other — the base national forecast does not equal the sum of the
# base store forecasts. Any business plan built on incoherent forecasts is internally
# inconsistent. We quantify the incoherence before vs after.
incoh_before = abs(y_hat[0] - y_hat[3:].sum())
incoh_after = abs(y_tilde[0] - y_tilde[3:].sum())
print(f'national vs sum-of-stores gap: base forecast {incoh_before:.3f}  ->  reconciled {incoh_after:.6f}')
assert incoh_before > 1e-3, 'independent base forecasts are incoherent (do not add up)'
assert incoh_after < 1e-6, 'reconciliation enforces the hierarchy exactly'
print('\nIndependent forecasts contradict the hierarchy -> reconcile (or forecast bottom-up) for consistency.')

## ✏️ Your turn — bottom-up vs top-down

In [ ]:
def bottom_up(bottom_forecasts, S):
    """Aggregate bottom-level forecasts to all levels using summing matrix."""
    # TODO(you): return y_tilde = S @ bottom_forecasts
    return ...

def top_down_proportional(top_forecast, S, historical_proportions):
    """Disaggregate top-level forecast using historical proportions."""
    # TODO(you): multiply top_forecast by each proportion, aggregate with S
    # historical_proportions: (m,) bottom-level proportions summing to 1
    return ...

# Test
hist_props = b_true / b_true.sum()
y_bu   = bottom_up(b_hat, S)
y_td   = top_down_proportional(y_hat[0], S, hist_props)

print("Bottom-up reconciled:", y_bu.round(2))
print("Top-down reconciled: ", y_td.round(2))
print("Bottom-up coherence error:", abs(y_bu[0] - y_bu[3:].sum()).round(6))
print("Top-down coherence error: ", abs(y_td[0] - y_td[3:].sum()).round(6))

<details><summary>Solution</summary>

```python
def bottom_up(bottom_forecasts, S):
    return S @ bottom_forecasts

def top_down_proportional(top_forecast, S, historical_proportions):
    bottom_level = top_forecast * historical_proportions
    return S @ bottom_level
```
</details>

## Key takeaways

- **Independent forecasts are incoherent:** levels don't add up (demo).
- **Reconciliation (MinT)** projects onto the coherent subspace — the hierarchy holds exactly
  (verified).
- **It improves accuracy** by pooling information across levels (verified).
- **Bottom-up** (sum the base series) is the simplest coherent alternative but ignores top-level
  signal.